<a href="https://colab.research.google.com/github/ivanChuhonin/LLM_RAG_TgBot/blob/main/LLM_RAG_TgBot_%D0%A7%D1%83%D1%85%D0%BE%D0%BD%D0%B8%D0%BD_%D0%9B%D0%A04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 4. Интерфейс, определение ИИ текста

**Задание**:
Для разработанной ранее модели реализовать интерфейсную часть.

Этапы выполнения лабораторной работы:

1. Для модели из лабораторной работы 1-3:

  a.  Реализовать интерфейсную часть на выбор студента:

 *   i. Телеграмм-бот

 *   ii. Бот в Discord

 *   iii. С помощью библиотек Python (Tkinter, PyQT, Yel)

2. Интерфейс должен содержать:

  a. Поле для ввода запроса

  b. Кнопка для отправки запроса

  c. Поле для получения ответа

3. Изучить способы определения текста, который сгенерировал ИИ
(см. ссылки). С помощью программных инструментов проверить
свои результаты и ответить на вопросы:
 *  Какие конструкции являются явными маркерами, что текст написан
ИИ? Привести примеры из своих тестов, явно показать такие места.
 *  Какие есть способы придать тексту больше «человечности»?

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/usr/local/lib/python3.11/dist-packages/torch_xla/__init__.py:253: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np
import string

In [ ]:
from typing import Dict, List, Optional

# Скачиваем данные

In [ ]:
!mkdir -p /root/.kaggle
!echo '{"username":"###","key":"###"}' > /root/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d abhishek/the-movie-dialog-dataset --force -f task3_qarecs_train.txt
!kaggle datasets download -d abhishek/the-movie-dialog-dataset --force -f task3_qarecs_test.txt
!kaggle datasets download -d abhishek/the-movie-dialog-dataset --force -f task1_qa_train.txt

Dataset URL: https://www.kaggle.com/datasets/abhishek/the-movie-dialog-dataset
License(s): Attribution 3.0 Unported (CC BY 3.0)
 97% 81.0M/83.1M [00:00<00:00, 154MB/s]
100% 83.1M/83.1M [00:00<00:00, 145MB/s]
Dataset URL: https://www.kaggle.com/datasets/abhishek/the-movie-dialog-dataset
License(s): Attribution 3.0 Unported (CC BY 3.0)
  0% 0.00/394k [00:00<?, ?B/s]
100% 394k/394k [00:00<00:00, 130MB/s]
Dataset URL: https://www.kaggle.com/datasets/abhishek/the-movie-dialog-dataset
License(s): Attribution 3.0 Unported (CC BY 3.0)
  0% 0.00/2.04M [00:00<?, ?B/s]
100% 2.04M/2.04M [00:00<00:00, 151MB/s]


In [ ]:
!unzip -q task3_qarecs_train.txt.zip
!unzip -q task3_qarecs_test.txt.zip
!unzip -q task1_qa_train.txt.zip

In [ ]:
df = pd.read_csv('/content/task3_qarecs_train.txt', sep='\t', names=['text', 'answer'])
df['text'] = df['text'].str[2:]
df.head(3)

,text,answer
0,"I really like Jaws, Bottle Rocket, Saving Priv...",Beyond the Mat
1,Who is that directed by?,Barry W. Blaustein
2,I like Jon Fauer movies more. Do you know anyt...,Cinematographer Style


In [ ]:
test_df = pd.read_csv('/content/task3_qarecs_test.txt', sep='\t', names=['text', 'answer'])
test_df['text'] = test_df['text'].str[2:]
# test_df.head(4)

In [ ]:
df_extra = pd.read_csv('/content/task1_qa_train.txt', sep='\t', names=['text', 'answer'])
df_extra['text'] = df_extra['text'].str[2:]
df_extra.head(4)

,text,answer
0,what movies are about ginger rogers?,"Top Hat, Kitty Foyle, The Barkleys of Broadway"
1,which movies can be described by moore?,"Fahrenheit 9/11, Far from Heaven"
2,what films can be described by occupation?,"Red Dawn, The Teahouse of the August Moon"
3,which films are about jacques tati?,"Mon Oncle, Playtime, Trafic"


In [ ]:
# Создаем маску
mask = [i%3 != 1 for i in range(len(df))]
# Удаляем строки, где маска равна False
df2 = df[mask]

In [ ]:
def remove_punctuation(text):
    punctuation_chars = string.punctuation
    mapping_table = str.maketrans('', '', punctuation_chars)
    return text.str.translate(mapping_table)


df_extra['text'] = remove_punctuation(df_extra['text'])
df2['text'] = remove_punctuation(df2['text'])

In [ ]:
df2 = df2.head(1000000)

In [ ]:
df2 = pd.concat([df2, df_extra], ignore_index=True)

In [ ]:
len(df2)

1096185

In [ ]:
mask = [i%3 != 1 for i in range(len(test_df))]
test_df2 = test_df[mask]

In [ ]:
df2[:12]

,text,answer
0,I really like Jaws Bottle Rocket Saving Privat...,Beyond the Mat
1,I like Jon Fauer movies more Do you know anyth...,Cinematographer Style
2,I loved Full Metal Jacket The Breakfast Club T...,Broadcast News
3,I like Jack Nicholson movies more Do you know ...,Carnal Knowledge
4,Edward Scissorhands Being John Malkovich To Ki...,The Breakfast Club
5,I rate Joel Schumacher movies Any other sugges...,St. Elmo's Fire
6,I liked Shallow Grave True Romance The Matrix ...,Bad Lieutenant
7,I rate Sidney Lumet movies Any other suggestions,Find Me Guilty
8,Bhaji on the Beach Strictly Ballroom Dancer in...,Eraserhead
9,I prefer zombie movies Can you suggest an alte...,Night of the Living Dead


In [ ]:
df2 = df2.head(1000000)

In [ ]:
len(df2)

1000000

In [ ]:
search_string = 'Russia'
result = df2[df2['text'].str.contains(search_string, case=False, na=False)]

In [ ]:
result[:20]

,text,answer
1192,I love Buena Vista Social Club Casino Royale T...,Casino Royale
1326,The Spanish Prisoner From Russia with Love To ...,The Monster
1620,I watched the films The Living Daylights Wayne...,On Her Majesty's Secret Service
1847,The Fugitive Psycho Gaslight Pulp Fiction Metr...,Anna Karenina
2829,I loved Hoop Dreams Braveheart A Goofy Movie B...,Burnt by the Sun
5266,The Fifth Element Office Space Bowling for Col...,Walk the Line
5557,I prefer Russian movies Can you suggest an alt...,Eastern Promises
5578,North by Northwest From Russia with Love Arsen...,A Day at the Races
5997,Eraser From Russia with Love Brazil The Purple...,The Specials
6022,I like russia movies more Do you know anything...,Rocky IV


# Реализация

In [ ]:
model_name = 'NousResearch/Llama-2-7b-chat-hf'

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
# Проверка текущей конфигурации генерации
print(model.generation_config)

# Доступ к конкретным параметрам
default_temperature = model.generation_config.temperature
default_top_k = model.generation_config.top_k

print(f"Temperature по умолчанию: {default_temperature}")
print(f"Top_k по умолчанию: {default_top_k}")

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 32000,
  "temperature": 0.9,
  "top_p": 0.6
}

Temperature по умолчанию: 0.9
Top_k по умолчанию: 50


In [ ]:
!pip install python-telegram-bot --upgrade
!pip install nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.1/676.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.9 MB/s eta 0:00:00


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from torch.utils.data import DataLoader
from tqdm import tqdm
import nest_asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes

Начальная реализация

In [ ]:
# Document Database Class
class DocumentDatabase:
    def __init__(self):
        self.documents = {}
        self.tfidf_vectorizer = None
        self.document_vectors = None
        self.doc_ids = []

    def add_document(self, doc_id, question, answer):
        self.documents[doc_id] = {"question": question, "answer": answer}
        self.doc_ids.append(doc_id)

    def get_document(self, doc_id):
        return self.documents.get(doc_id, {"question": "", "answer": ""})

    def build_index(self):
        questions = [self.documents[doc_id]["question"] for doc_id in self.doc_ids]
        self.tfidf_vectorizer = TfidfVectorizer().fit(questions)
        document_vectors = self.tfidf_vectorizer.transform(questions)
        self.document_vectors = csr_matrix(document_vectors)

    def search(self, query):
        query_vector = self.tfidf_vectorizer.transform([query])
        similarities = query_vector.dot(self.document_vectors.T).toarray()[0]
        best_match_idx = np.argmax(similarities)
        best_match_score = similarities[best_match_idx]
        if best_match_score > 0:
            return self.doc_ids[best_match_idx]
        else:
            return None

# Retrieve Relevant Document
def retrieve_relevant_document(query, database):
    relevant_doc_id = database.search(query)
    if relevant_doc_id:
        return database.get_document(relevant_doc_id)["answer"]
    else:
        return ""

# Generate Answer
def generate_answer(question, answer, model, tokenizer, max_length=120):
    input_text = f"Question: {question}\nAnswer: {answer}"
    inputs = tokenizer(input_text, return_tensors="pt") #.to(device)
    outputs = model.generate(**inputs, max_length=max_length, top_k=20, temperature=0.3)
    generated_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_answer = generated_answer.rsplit(".", 1)[0]
    return generated_answer.split("\n")[1]

# Main Function to Initialize Database
def initialize_database():

    database = DocumentDatabase()
    for doc_id, example in df2.iterrows():
        question = example['text']
        answer = example['answer']
        database.add_document(doc_id, question, answer)

    database.build_index()
    return database

# Initialize the database
database = initialize_database()

# Telegram Bot Handlers
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Hi! I'm a bot that can recommend movies based on your wishes. Let's try it.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_message = update.message.text
    # Retrieve relevant document
    retrieved_answer = retrieve_relevant_document(user_message, database)
    print(retrieved_answer)
    # Generate answer using the retrieved document
    predicted_answer = generate_answer(user_message, retrieved_answer, model, tokenizer)
    # Send the generated answer to the user
    await update.message.reply_text(predicted_answer)

# Main Function to Run the Bot
async def main():
    application = ApplicationBuilder().token('###').build()
    application.add_handler(CommandHandler("start", start))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    await application.run_polling()

# Run the bot
if __name__ == "__main__":
    nest_asyncio.apply()
    import asyncio
    asyncio.run(main())

The Prestige, Labyrinth, The Hunger, Basquiat, The Man Who Fell to Earth, Absolute Beginners
Pusher
Glengarry Glen Ross
Fear and Loathing in Las Vegas


RuntimeError: Cannot close a running event loop

Финальная версия

In [ ]:
class DocumentDatabase:
    def __init__(self):
        self.documents: Dict[int, Dict[str, str]] = {}
        self.tfidf_vectorizer: Optional[TfidfVectorizer] = None
        self.document_vectors: Optional[csr_matrix] = None
        self.doc_ids: List[int] = []

    def add_document(self, doc_id: int, question: str, answer: str) -> None:
        """Add a document to the database."""
        self.documents[doc_id] = {"question": question, "answer": answer}
        self.doc_ids.append(doc_id)

    def get_document(self, doc_id: int) -> Dict[str, str]:
        """Retrieve a document by its ID."""
        return self.documents.get(doc_id, {"question": "", "answer": ""})

    def build_index(self) -> None:
        """Build the TF-IDF index for the documents."""
        questions = [self.documents[doc_id]["question"] for doc_id in self.doc_ids]
        self.tfidf_vectorizer = TfidfVectorizer(ngram_range = (1, 2)).fit(questions)
        self.document_vectors = csr_matrix(self.tfidf_vectorizer.transform(questions))

    def search(self, query: str, threshold: float = 0.2) -> Optional[int]:
        """Search for the most relevant document."""
        query_vector = self.tfidf_vectorizer.transform([query])
        similarities = query_vector.dot(self.document_vectors.T).toarray()[0]
        best_match_idx = np.argmax(similarities)
        best_match_score = similarities[best_match_idx]
        return self.doc_ids[best_match_idx] if best_match_score > threshold else None

def retrieve_relevant_document(query: str, database: DocumentDatabase) -> str:
    """Retrieve the most relevant document for a query."""
    relevant_doc_id = database.search(query)
    return database.get_document(relevant_doc_id)["answer"] if relevant_doc_id else ""

def generate_answer(question: str, answer: str, model, tokenizer, max_length: int = 120) -> str:
    """Generate an answer using a pre-trained model."""
    input_text = f"Question: {question}\n {answer}"
    inputs = tokenizer(input_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=max_length, top_k=20, temperature=0.2)
    generated_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_answer = generated_answer.rsplit(".", 1)[0]
    print(generated_answer)
    if not generated_answer.endswith("."):
        generated_answer += "."
    return generated_answer.split("\n")[1]

def initialize_database() -> DocumentDatabase:
    """Initialize the database with documents."""
    database = DocumentDatabase()
    for doc_id, example in df.iterrows():
        database.add_document(doc_id, example['text'], example['answer'])
    database.build_index()
    return database

# Telegram Bot Handlers
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    await update.message.reply_text("Hi! I'm a bot that can recommend movies based on your wishes. Let's try it.")

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    help_text = "Available commands:\n/start\n/help"
    await update.message.reply_text(help_text)

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    await update.message.reply_text("Accepted! Wait for response...")
    user_message = update.message.text
    retrieved_answer = retrieve_relevant_document(user_message, database)
    print(retrieved_answer)
    predicted_answer = generate_answer(user_message, retrieved_answer, model, tokenizer)
    await update.message.reply_text(predicted_answer)

async def main() -> None:
    """Run the Telegram bot."""
    application = ApplicationBuilder().token('###').build()
    application.add_handler(CommandHandler("start", start))
    application.add_handler(CommandHandler("help", help_command))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    await application.run_polling()

if __name__ == "__main__":
    nest_asyncio.apply()
    database = initialize_database()
    import asyncio
    asyncio.run(main())

Saraband
Question: I rate Ingmar Bergman movies. Any other suggestions?
Saraband (2003)
Fanny and Alexander (1982)
Persona (1966)
The Seventh Seal (1957)
Wild Strawberries (1957)

Answer: Certainly! Ingmar Bergman is a legendary filmmaker known for his thought-provoking and emotionally charged films. Here are some other suggestions:

1
Only Lovers Left Alive
Question: I want a movie with a hilarious dialogues and breathtaking shootouts. I really like Jim Jarmusch movies. Any suggestions?
Only Lovers Left Alive (2013) - This vampire romance film directed by Jim Jarmusch features a hilarious dialogue between the two vampire lovers, played by Tilda Swinton and Tom Hiddleston. The film also has some breathtaking shootouts, especially in the climax
The Pursuit of Happyness
Question: Forrest Gump, The Wizard of Oz, The Little Mermaid, Moulin Rouge!, Shrek 2, Beauty and the Beast, and Life Is Beautiful are movies I really like. I'm looking for a Mystery movie.
The Pursuit of Happyness, The Bl

RuntimeError: Cannot close a running event loop

Оценка работы

In [ ]:
text = "Only Lovers Left Alive (2013) is a great choice for you! It's a vampire romance film directed by Jim Jarmusch, known for his unique and quirky sense of humor. The dialogue is witty and engaging, and the shootouts are stylish and well-choreographed."

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Get the logits (raw model outputs)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Calculate the loss (cross-entropy loss)
loss_fn = torch.nn.CrossEntropyLoss()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = inputs["input_ids"][:, 1:].contiguous()
loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

# Calculate perplexity
perplexity = torch.exp(loss).item()

print("Perplexity:", perplexity)

Perplexity: 4.414498805999756


In [ ]:
text = "Burnt by the Sun (1994) is a great Russian movie that you might enjoy. It's a historical drama directed by Nikita Mikhalkov and it won the Palme d'Or at the 1994 Cannes Film Festival."

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Get the logits (raw model outputs)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Calculate the loss (cross-entropy loss)
loss_fn = torch.nn.CrossEntropyLoss()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = inputs["input_ids"][:, 1:].contiguous()
loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

# Calculate perplexity
perplexity = torch.exp(loss).item()

print("Perplexity:", perplexity)

Perplexity: 2.5231165885925293


In [ ]:
text = "Half Nelson (2006) and Drive (2011) are two movies starring Ryan Gosling that are known for their intense action and suspenseful plots. Half Nelson follows a drug-addicted high school teacher who forms an unlikely bond with one of his students, while Drive is a crime drama about a stunt driver who moonlights as a getaway driver for criminals."

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Get the logits (raw model outputs)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Calculate the loss (cross-entropy loss)
loss_fn = torch.nn.CrossEntropyLoss()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = inputs["input_ids"][:, 1:].contiguous()
loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

# Calculate perplexity
perplexity = torch.exp(loss).item()

print("Perplexity:", perplexity)

Perplexity: 2.4184253215789795


In [ ]:
text = "Schindler's List, The Shawshank Redemption, The Godfather are highly rated thriller movies that you might enjoy."

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Get the logits (raw model outputs)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Calculate the loss (cross-entropy loss)
loss_fn = torch.nn.CrossEntropyLoss()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = inputs["input_ids"][:, 1:].contiguous()
loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

# Calculate perplexity
perplexity = torch.exp(loss).item()

print("Perplexity:", perplexity)

Perplexity: 8.221983909606934


In [ ]:
text = 'Roy William Neill directed the majority of the 23 films in the "Hammer Horror" series, including "The Hound of the Baskervilles" (1959), "The Mummy" (1959), "The Brides of Fu Manchu" (1965)'

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Get the logits (raw model outputs)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Calculate the loss (cross-entropy loss)
loss_fn = torch.nn.CrossEntropyLoss()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = inputs["input_ids"][:, 1:].contiguous()
loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

# Calculate perplexity
perplexity = torch.exp(loss).item()

print("Perplexity:", perplexity)

Perplexity: 2.238807201385498


In [ ]:
import torch.nn as nn

# Определите функцию потерь
loss_fn = nn.CrossEntropyLoss()

def calculate_perplexity(model, tokenizer, dataset):
    total_loss = 0
    total_tokens = 0
    for text in dataset:
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        # Calculate loss
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = inputs["input_ids"][:, 1:].contiguous()
        loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        total_loss += loss.item() * shift_labels.numel()
        total_tokens += shift_labels.numel()
    # Calculate average perplexity
    avg_loss = total_loss / total_tokens
    avg_perplexity = torch.exp(torch.tensor(avg_loss)).item()
    return avg_perplexity

dataset = [
    "Burnt by the Sun is a great choice. It's a 1994 film directed by Nikita Mikhalkov, and it's a powerful and thought-provoking drama that explores the themes of war, loss, and redemption. The film is set during World War II and follows a group of Soviet soldiers who are sent on a mission to investigate a series of strange occurrences in a remote village.",
    "Only Lovers Left Alive (2013) - This vampire romance film directed by Jim Jarmusch features a hilarious dialogue between the two vampire lovers, played by Tilda Swinton and Tom Hiddleston. The film also has some breathtaking shootouts, especially in the climax.",
    "Collateral Damage (2002) - Arnold Schwarzenegger, Frank Langella, and Heather Graham star in this action-packed thriller about a firefighter who must stop a terrorist from detonating a bomb in Los Angeles."
]

# Рассчитайте среднюю перплексию
avg_perplexity = calculate_perplexity(model, tokenizer, dataset)
print("Average Perplexity:", avg_perplexity)

Average Perplexity: 3.0356757640838623


# Проверка определения текста, который сгенерировал ИИ

In [ ]:
import requests

key = 'B9CEMZN4ZQNF6NJVX1DHAY5T0SS5W29U'
url = 'https://api.sapling.ai/api/v1/aidetect'
data = {
    'key': key,
    'text': "Big Fish (2003) is a great movie that combines elements of thriller, drama, and fantasy. It's a dark and emotional film that explores the themes of family, love, and the power of story",
}

try:
    resp = requests.post(url, json=data)
    resp_json = resp.json()

    if 200 <= resp.status_code < 300:
        score = resp_json['score']
        print('Score: ', score)
    else:
        print('Error: ', resp_json)

except Exception as e:
    print('Error: ', e)

Score:  0.9999981328497402


In [ ]:
data = {
    'key': key,
    'text': "Jackie Brown, Kill Bill, and Death Proof are all great options for action-packed Tarantino movies with plenty of blood and violence. Here's why",
}

try:
    resp = requests.post(url, json=data)
    resp_json = resp.json()

    if 200 <= resp.status_code < 300:
        score = resp_json['score']
        print('Score: ', score)
    else:
        print('Error: ', resp_json)

except Exception as e:
    print('Error: ', e)

Score:  0.9999799717955311


In [ ]:
data = {
    'key': key,
    'text': "It's a Wonderful Life (1946) is a classic Christmas movie that has been enjoyed by many for generations. It is a heartwarming tale of a man who is given the chance to see what the world would be like if he had never been born. The movie stars James Stewart",
}

try:
    resp = requests.post(url, json=data)
    resp_json = resp.json()

    if 200 <= resp.status_code < 300:
        score = resp_json['score']
        print('Score: ', score)
    else:
        print('Error: ', resp_json)

except Exception as e:
    print('Error: ', e)

Score:  0.9999896170675823


In [ ]:
data = {
    'key': key,
    'text': "Andrei Rublev (1966) is a classic Russian film directed by Andrei Tarkovsky that explores the life of a renowned",
}

try:
    resp = requests.post(url, json=data)
    resp_json = resp.json()

    if 200 <= resp.status_code < 300:
        score = resp_json['score']
        print('Score: ', score)
    else:
        print('Error: ', resp_json)

except Exception as e:
    print('Error: ', e)

Score:  0.9999953749008991


# Выводы

1. Какие конструкции являются явными маркерами, что текст написан
ИИ? Привести примеры из своих тестов, явно показать такие места.

 Примеры: is a great choice, is a highly acclaimed, here is why + часто получается что второе предложение начинается с It is

2. Какие есть способы придать тексту больше «человечности»?

*   Редактирование и доработка вручную;
*   Использовать персонализированные GPT;
*   Использование специализированных инструментов.


